# Multi-Area RNN — Kleinman et al. 2025 (eLife), Paradigm A

> Kleinman, M., Wang, T., Xiao, D., ... Chandrasekaran, C., Kao, J.C. (2025).
> *The information bottleneck as a principle underlying multi-area cortical
> representations during decision-making.* eLife. DOI: 10.7554/eLife.89369.2

When monkeys perform a perceptual decision task, upstream cortex (DLPFC)
represents all task variables while downstream cortex (PMd) keeps only what
is needed for the action — a *minimal sufficient representation*. This paper
shows that a multi-area RNN trained on the same task reproduces that
phenomenon spontaneously: information about the choice direction survives
across areas, while information about color and target configuration is
progressively dropped (an information-bottleneck effect).

In this notebook we (no public code exists for the paper, so this is a
model-only reproduction):

1. implement the paper's **checkerboard decision task**;
2. build the paper's **3-area cascaded RNN** in two equivalent ways —
   the `multiarea_rnn` family, and a plain `constrained_rnn` with hand-built
   block masks — and verify they are identical;
3. train it with the framework's standard `Trainer`;
4. look at the **neural trajectories of the three areas** and quantify how
   much each area knows about each task variable;
5. examine **why** direction information is propagated downstream while other
   variables are not.

## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

sys.path.insert(0, str(Path.cwd().parent / "src"))

from neuralrnn import AutoModel, Trainer, TrainingArguments, SupervisedObjective
from neuralrnn.models.multiarea_rnn import (
    MultiAreaRNNConfig, build_multiarea_masks,
    rescale_effective_spectral_radius)
from neuralrnn.models.constrained_rnn import (
    ConstrainedRNNConfig, ConstrainedRNNModel)
from neuralrnn.data.cognitive_task_dataset import CognitiveTaskDataset
from neuralrnn.data.tasks import checkerboard_trials
from neuralrnn.analysis import (
    fit_pca, fit_dpca, axis_overlap_matrix, axis_svd_alignment,
    potent_null_projection)
from neuralrnn.visualization import plot_weight_matrix

DEVICE = "cpu"  # set to "cuda" for faster training if a GPU is available
MODEL_DIR = Path("./models/17")
FIG_DIR = Path("./figs/17")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters (paper Table 1 where stated; adjusted for fast convergence)
AREA_SIZES = [100, 100, 100]  # 3 areas x 100 units
DT_MS, TAU_MS = 10.0, 50.0
LR, BATCH_SIZE, MAX_STEPS = 1e-3, 128, 2000
SIGMA_REC = 0.1               # recurrent noise std
DECISION_THRESHOLD = 0.7      # DV threshold for reading out the decision
N_EVAL = 2000                 # eval trials for behavior / analyses
print("device:", DEVICE)

## 1. The checkerboard task

A red/green checkerboard is shown and the network must report the majority
color by "reaching" to the matching target. Two targets (left/right) appear
each trial with randomized colors, so the **color** decision and the
**direction** decision are statistically independent, and the target
configuration (which side shows which color) is a task variable that does not
need to be reported.

- **Inputs (4):** left/right target colors (±1, on during the targets epoch),
  red/green signed coherences (on during the decision epoch, with
  N(0, 0.1²) noise).
- **Outputs (2):** decision variables (DVs) for left/right reach; the correct
  DV should ramp to 1 during the decision epoch. The first 200 ms of the
  decision epoch are excluded from the loss (integration ramp).
- 10% of trials are catch trials (no decision required).
- `mode="eval"` fixes the epoch durations so trials align for analysis.

In [ ]:
inputs, targets, mask, conditions = checkerboard_trials(
    n_trials=8, mode="eval", seed=42)
print("inputs", tuple(inputs.shape), "targets", tuple(targets.shape))
print("channels: [left_target_color, right_target_color, red_coh, green_coh]")
print("condition example:", {k: v for k, v in conditions[0].items() if k != "epoch_bounds"})

fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
t = np.arange(inputs.shape[1]) * DT_MS
for ch, name in enumerate(["left tgt color", "right tgt color", "red coh", "green coh"]):
    axes[0].plot(t, inputs[0, :, ch], label=name)
axes[0].legend(fontsize=8); axes[0].set_ylabel("input")
for ch, name in enumerate(["left DV", "right DV"]):
    axes[1].plot(t, targets[0, :, ch], label=name)
axes[1].fill_between(t, 0, 1, where=mask[0, :, 0] == 0, alpha=0.2, color="gray",
                     label="loss mask off")
axes[1].legend(fontsize=8); axes[1].set_ylabel("target"); axes[1].set_xlabel("time (ms)")
for ax in axes:
    for b in conditions[0]["epoch_bounds"]:
        ax.axvline(b * DT_MS, ls=":", c="k", lw=0.8)
fig.suptitle("Example checkerboard trial (eval timing)")
fig.tight_layout()
fig.savefig(FIG_DIR / "task_example.png", dpi=150)
plt.show()

## 2. Building the multi-area RNN (two equivalent ways)

**Architecture** (paper Table 1): 300 units in 3 cascaded areas of 100, Dale's
law with 80 excitatory / 20 inhibitory units *per area*, dense intra-area
recurrence, and inter-area projections that **originate only from excitatory
units** (long-range cortical projections are excitatory): 10% feedforward
E→E, 2% feedforward E→I, 5% feedback E→E. Input enters Area 1 and the readout
reads the E units of Area 3. τ = 50 ms, dt = 10 ms, ReLU, recurrent noise.

- **Path A (family):** `MultiAreaRNNConfig` generates all masks and the
  per-area Dale sign vector from these area-level hyperparameters, and
  rescales the effective recurrent matrix to spectral radius 1.0 at init
  (the framework-default init is unstable under Dale's law over long trials).
- **Path B (manual):** the same masks, built with `build_multiarea_masks`,
  passed to a plain `ConstrainedRNNConfig` — this is all the family does
  internally.

Given the same weights, the two paths are bit-exact (asserted below).

In [ ]:
def build_multiarea_model(seed=0, ff_ei_density=0.02, fb_density=0.05):
    """Path A: the multiarea_rnn family."""
    cfg = MultiAreaRNNConfig(
        input_dim=4, output_dim=2, area_sizes=AREA_SIZES,
        dt=DT_MS, tau=TAU_MS, activation="relu", sigma_rec=SIGMA_REC,
        ff_ei_density=ff_ei_density, fb_density=fb_density, mask_seed=seed)
    return AutoModel.from_config(cfg)


def build_manual_constrained_model(seed=0, ff_ei_density=0.02, fb_density=0.05):
    """Path B: constrained_rnn with hand-built block masks."""
    rec, in_m, out_m, signs = build_multiarea_masks(
        area_sizes=AREA_SIZES, input_dim=4, output_dim=2,
        ff_ei_density=ff_ei_density, fb_density=fb_density, mask_seed=seed)
    cfg = ConstrainedRNNConfig(
        input_dim=4, latent_dim=sum(AREA_SIZES), output_dim=2,
        dt=DT_MS, tau=TAU_MS, activation="relu", sigma_rec=SIGMA_REC,
        dale=True, dale_signs=signs.tolist(),
        rec_mask=rec, in_mask=in_m, out_mask=out_m)
    model = ConstrainedRNNModel(cfg)
    rescale_effective_spectral_radius(model, 1.0)  # same init as the family
    return model


model_a = build_multiarea_model(seed=0).to(DEVICE)
model_b = build_manual_constrained_model(seed=0).to(DEVICE)

# parity check with identical weights
with torch.no_grad():
    for pa, pb in zip(model_a.parameters(), model_b.parameters()):
        pb.copy_(pa)
model_a.eval(); model_b.eval()
x = torch.randn(2, 50, 4, device=DEVICE)
assert (model_a(x).states - model_b(x).states).abs().max().item() == 0.0
print("Path A == Path B (bit-exact rollout parity)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_weight_matrix(model_a.rec_mask.cpu().numpy(), ax=axes[0], 
                   cmap='Greys', center_zero=False,
                   title="rec_mask: 3-area cascade")
signs = np.diag(model_a.dale_mask.cpu().numpy())
axes[1].imshow(signs.reshape(3, 100), cmap="RdBu_r", aspect="auto", vmin=-1, vmax=1)
axes[1].set_xlabel("unit within area"); axes[1].set_ylabel("area")
axes[1].set_title("Dale signs: per-area 80 E (+1) / 20 I (-1)")
fig.tight_layout()
fig.savefig(FIG_DIR / "architecture_masks.png", dpi=150)
plt.show()

## 3. Training

Standard framework training: masked-MSE regression on the two DVs
(`SupervisedObjective`), Adam with lr 1e-3, batch 128, 2000 steps. A
`post_step_hook` re-projects the weights onto the mask support after every
step (masked weights stay exactly zero). Checkpoints are load-first: rerunning
this notebook loads `models/17/3area_seed0` instead of retraining.

`evaluate_decision` reads out behavior the way the paper does: the decision
is the first DV that crosses 0.7 during the decision epoch (falling back to
the larger DV at epoch end).

In [ ]:
@torch.no_grad()
def evaluate_decision(model, n_trials=N_EVAL, seed=123):
    """Decision = first DV crossing the threshold during the decision epoch."""
    model.eval()
    inp, _, _, cond = checkerboard_trials(
        n_trials=n_trials, mode="eval", balanced=True, seed=seed)
    dvs = model(inp.to(DEVICE)).outputs.cpu().numpy()
    recs = []
    for i, c in enumerate(cond):
        if c["catch"]:
            continue
        b = c["epoch_bounds"]
        dv = dvs[i, b[1]:b[2]]
        over = np.argwhere(dv > DECISION_THRESHOLD)
        if len(over):
            t_first = over[0, 0]
            choice = int(dv[t_first, 1] > dv[t_first, 0])
            rt = t_first
        else:
            choice, rt = int(dv[-1, 1] > dv[-1, 0]), None
        recs.append({"correct_choice": c["correct_choice"], "choice": choice,
                     "coherence": c["coherence"], "left_color": c["left_color"],
                     "rt": rt, "correct": int(choice == c["correct_choice"])})
    acc_l = np.mean([r["correct"] for r in recs if r["correct_choice"] == 0])
    acc_r = np.mean([r["correct"] for r in recs if r["correct_choice"] == 1])
    model.train()
    return float(acc_l), float(acc_r), recs


def train_or_load(seed=0, overwrite=False):
    """Train the 3-area network, or load the checkpoint if it exists."""
    save_dir = MODEL_DIR / f"3area_seed{seed}"
    if (save_dir / "model.safetensors").exists() and not overwrite:
        return AutoModel.from_pretrained(save_dir).to(DEVICE), save_dir, None
    model = build_multiarea_model(seed=seed).to(DEVICE)
    ds = CognitiveTaskDataset.from_task("checkerboard", batch_size=BATCH_SIZE)
    args = TrainingArguments(max_steps=MAX_STEPS, learning_rate=LR,
                             grad_clip_norm=1.0, log_every=100,
                             device=DEVICE, seed=seed)
    trainer = Trainer(model, ds, SupervisedObjective(task_type="regression"),
                      args,
                      post_step_hook=lambda m: m._apply_masks_to_weights())
    log = trainer.train()
    acc_l, acc_r, _ = evaluate_decision(model)
    print(f"final accuracy L/R = {acc_l:.3f}/{acc_r:.3f}")
    save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_dir)
    return model, save_dir, log

In [ ]:
model, save_dir, log = train_or_load(seed=0)
model.eval()
print("exemplar network:", save_dir)

if log:  # loss curve, only when (re)trained this session
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot([h["step"] for h in log], [h["loss"] for h in log], "o-", ms=3)
    ax.set_xlabel("step"); ax.set_ylabel("masked MSE")
    ax.set_title("Training curve")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "training_curve.png", dpi=150)
    plt.show()

## 4. Behavior

The trained network's behavior should look like the animals': a sigmoidal
psychometric curve (probability of choosing right vs signed coherence) and
faster decisions at higher coherence.

In [ ]:
_, _, recs = evaluate_decision(model, n_trials=N_EVAL, seed=7)
cohs = np.array([r["coherence"] for r in recs])
chose_right = np.array([r["choice"] for r in recs])
rts = np.array([r["rt"] if r["rt"] is not None else np.nan for r in recs])
left_color = np.array([r["left_color"] for r in recs])
signed = np.where(left_color < 0, -cohs, cohs)  # >0: right target color is majority

bins = np.linspace(-1, 1, 15)
centers = 0.5 * (bins[:-1] + bins[1:])
def binned(vals, x):
    return np.array([vals[(x >= lo) & (x < hi)].mean()
                     if ((x >= lo) & (x < hi)).any() else np.nan
                     for lo, hi in zip(bins[:-1], bins[1:])])

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(centers, binned(chose_right.astype(float), signed), "o-")
axes[0].set_xlabel("signed coherence (toward right)"); axes[0].set_ylabel("P(choose right)")
axes[0].set_ylim(0, 1); axes[0].set_title("psychometric")
axes[1].plot(centers, binned(rts, np.abs(cohs)) * DT_MS, "o-", c="tab:orange")
axes[1].set_xlabel("|coherence|"); axes[1].set_ylabel("RT (ms)"); axes[1].set_title("chronometric")
fig.suptitle("RNN behavior")
fig.tight_layout(); fig.savefig(FIG_DIR / "behavior.png", dpi=150)
plt.show()

## 5. What does each area represent?

Now the central question: how do representations differ across the three
areas? We run eval trials, collect each area's activity, and look at three
things:

1. **Trajectories** — condition-averaged activity projected onto each area's
   top 3 PCs, colored by direction and color.
2. **dPCA relative variance** — demixed PCA (`analysis.demixed.fit_dpca`)
   splits each area's variance into components tied to direction, color, and
   target configuration.
3. **Decoding accuracy and usable information** — can a linear decoder read
   each variable out of each area (5-fold logistic regression on the mean
   activity over the last 500 ms)? Usable information = 1 bit − decoding
   cross-entropy.

The information-bottleneck signature: direction stays prominent everywhere,
while color and target configuration fade across the cascade.

In [ ]:
def run_eval_trials(model, n_trials=N_EVAL, seed=7):
    inp, _, _, cond = checkerboard_trials(
        n_trials=n_trials, mode="eval", balanced=True, seed=seed)
    with torch.no_grad():
        states = model(inp.to(DEVICE)).states.cpu().numpy()
    keep, acond = [], []
    for i, c in enumerate(cond):
        if c["catch"] or c["correct_choice"] < 0:
            continue
        keep.append(i)
        acond.append({
            "direction": c["correct_choice"],
            "color": int(c["coherence"] > 0),
            "target_config": int(c["left_color"] > 0)})
    bounds = next(c["epoch_bounds"] for c in cond if not c["catch"])
    return states, keep, acond, bounds


states, keep, acond, bounds = run_eval_trials(model)
tg_end, dec_end = bounds[1], bounds[2]
print(f"{len(keep)} analysis trials, T={states.shape[1]}, "
      f"decision = steps {tg_end}..{dec_end}")

In [ ]:
# condition-averaged trajectories in each area's top-3 PC space
fig = plt.figure(figsize=(9, 3))
for a, sl in enumerate(model.area_slices):
    X = states[keep][:, :, sl]
    pca = fit_pca(X.reshape(-1, X.shape[-1]), n_components=3)
    ax = fig.add_subplot(1, 3, a + 1, projection="3d")
    for d, c, color, lab in [
        (0, 0, "tab:blue", "left, green"), (1, 0, "tab:red", "right, red"),
        (1, 1, "tab:orange", "right, green"), (0, 1, "tab:green", "left, red")
        ]:
        idx = [i for i, cond in enumerate(acond)
               if cond["direction"] == d and cond["color"] == c]
        traj = pca.transform(X[idx].mean(0))
        ax.plot(traj[tg_end:dec_end, 0], traj[tg_end:dec_end, 1],
                traj[tg_end:dec_end, 2], color=color, label=lab)
        ax.scatter(*traj[tg_end, :3], color=color, marker="o")
        ax.scatter(*traj[dec_end - 1, :3], color=color, marker="x")
    ax.set_title(f"Area {a+1} (3 PCs = {pca.explained_variance_ratio.sum():.0%} var)")
    ax.legend(fontsize=8)
fig.suptitle("Per-area PC trajectories during the decision epoch")
fig.tight_layout(); fig.savefig(FIG_DIR / "pc_trajectories.png", dpi=150)
plt.show()

In [ ]:
# dPCA relative variance, decoding accuracy, usable information per area
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

VARS3 = ("direction", "color", "target_config")
dpca_res, dpca_var, dec_acc, usable = {}, {}, {}, {}
for a, sl in enumerate(model.area_slices):
    X = states[keep][:, :, sl]
    res = fit_dpca(X, acond, variables=VARS3, n_axes=1)
    dpca_res[a], dpca_var[a] = res, res.variance_ratio
    feat = X[:, -50:, :].mean(1)  # mean over the last 500 ms
    for var in VARS3:
        y = np.array([c[var] for c in acond])
        proba = cross_val_predict(LogisticRegression(max_iter=2000), feat, y,
                                  cv=5, method="predict_proba")
        dec_acc.setdefault(var, []).append(float((proba.argmax(1) == y).mean()))
        ce = -np.mean(np.log2(np.clip(proba[np.arange(len(y)), y], 1e-12, 1)))
        usable.setdefault(var, []).append(max(0.0, 1.0 - ce))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
xpos = np.arange(3)
for k, var in enumerate(VARS3):
    axes[0].bar(xpos + k * 0.25, [dpca_var[a][var] for a in range(3)], 0.25, label=var)
    axes[1].plot(xpos, dec_acc[var], "o-", label=var)
    axes[2].plot(xpos, usable[var], "o-", label=var)
axes[0].set_xticks(xpos + 0.25, [f"Area {a+1}" for a in range(3)])
axes[0].set_ylabel("dPCA relative variance"); axes[0].legend(fontsize=8)
for ax, ylab in [(axes[1], "decoding accuracy"), (axes[2], "usable information (bits)")]:
    ax.set_xticks(xpos, [f"Area {a+1}" for a in range(3)])
    ax.set_ylabel(ylab); ax.legend(fontsize=8)
axes[1].axhline(0.5, ls="--", c="gray"); axes[1].set_ylim(0.4, 1.02)
axes[2].set_ylim(0, 1.05)
fig.suptitle("Direction survives across areas; color / target-config are dropped")
fig.tight_layout(); fig.savefig(FIG_DIR / "area_information.png", dpi=150)
plt.show()

print("dPCA relative variance (direction / color / target_config):")
for a in range(3):
    v = dpca_var[a]
    print(f"  Area {a+1}: {v['direction']:.3f} / {v['color']:.3f} / {v['target_config']:.3f}")
print("decoding accuracy:")
for var in VARS3:
    print(f"  {var}: {[round(u, 3) for u in dec_acc[var]]}")
print("usable info (bits):")
for var in VARS3:
    print(f"  {var}: {[round(u, 3) for u in usable[var]]}")

## 6. Why is direction information propagated, but not color?

The paper's mechanism: in Area 1 the direction axis **partially
orthogonalizes** from the other variables and **preferentially aligns with
the top right singular vectors of the inter-area weight matrix W21** — i.e.
with the directions that Area 2 actually reads from Area 1. The color and
target-config axes align only about as much as random vectors, so they are
attenuated across the cascade.

Below we check, for the Area-1 dPCA axes (restricted to the E units, which
are the only ones projecting downstream):

- dot products between the axes (partial orthogonalization);
- |cos| of each axis with W21's right singular vectors, against a
  random-vector baseline;
- how much of each axis lies in the potent vs null space of Area 1's own
  recurrent matrix.

In [ ]:
# effective (masked, Dale-signed) recurrent weights
W_eff = (model._recurrent_weight()).detach().cpu().numpy()
s0, s1 = model.area_slices[0], model.area_slices[1]
e0 = np.arange(s0.start, s0.start + 80)  # Area-1 E units
e1 = np.arange(s1.start, s1.start + 80)  # Area-2 E units
W21 = W_eff[np.ix_(e1, e0)]
W11 = W_eff[np.ix_(np.arange(s0.start, s0.stop), np.arange(s0.start, s0.stop))]

res1 = dpca_res[0]
dir_ax, col_ax, tgt_ax = (res1.axes["direction"][0], res1.axes["color"][0],
                          res1.axes["target_config"][0])

# partial orthogonalization between Area-1 axes
print("Area-1 axis dot products:")
print(f"  dir x color  = {axis_overlap_matrix(dir_ax, col_ax)[0, 0]: .3f}")
print(f"  dir x target = {axis_overlap_matrix(dir_ax, tgt_ax)[0, 0]: .3f}")
print(f"  col x target = {axis_overlap_matrix(col_ax, tgt_ax)[0, 0]: .3f}")

# alignment with W21 right singular vectors (axes restricted to E units)
al_dir = axis_svd_alignment(dir_ax[:80], W21)
al_col = axis_svd_alignment(col_ax[:80], W21)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(al_dir["alignments"][:20], "o-", label="direction axis")
ax.plot(al_col["alignments"][:20], "s-", label="color axis")
ax.axhline(al_dir["random_mean"].mean(), ls="--", c="gray", label="random baseline")
ax.set_xlabel("W21 right singular vector index"); ax.set_ylabel("|cos| with axis")
ax.legend()
ax.set_title("Direction axis aligns with the top readout dimensions of W21")
fig.tight_layout(); fig.savefig(FIG_DIR / "w21_alignment.png", dpi=150)
plt.show()

# potent/null projection onto Area-1 W_rec (rank-10 potent space)
for name, ax_v in [("direction", dir_ax), ("color", col_ax)]:
    pn = potent_null_projection(ax_v, W11, rank=10)
    print(f"W11 rank-10 potent/null — {name}: "
          f"{pn['potent_frac']:.3f} / {pn['null_frac']:.3f}")

## Summary

- The 3-area cascade learns the checkerboard task with animal-like behavior
  (sigmoidal psychometric curve, coherence-dependent RT).
- **Representations differ across areas**: direction information is preserved
  across the cascade, while color and target-configuration variance and
  decodability drop — the downstream area keeps a (nearly) minimal sufficient
  representation, matching the paper's information-bottleneck account of
  DLPFC → PMd.
- **Mechanism**: the Area-1 direction axis partially orthogonalizes from the
  other task axes and aligns with the top right singular vectors of the
  inter-area weights W21, so it is preferentially propagated downstream;
  the color axis aligns only at the random-vector level.

Implementation notes:
- The network was built twice — via the `multiarea_rnn` family and via
  hand-built masks on `constrained_rnn` — with bit-exact parity.
- Deviations from the paper (details in `docs/papers/multi-area_rnn.md`):
  the recurrent init is rescaled to spectral radius 1.0 (the framework
  default is unstable under Dale's law), training uses the plain masked-MSE
  objective with lr 1e-3 (the paper's exact regularizers are not needed for
  the phenomenon; the paper shows it is robust to them), and decoders are
  logistic regression instead of small MLPs.
- Not reproduced: all comparisons against the DLPFC/PMd neural recordings
  (the data is not public).